In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# Saving the Infidels

In this notebook we want so solve a famous search problem, which is usually known as the
[missionaries and cannibals problem](https://en.wikipedia.org/wiki/Missionaries_and_cannibals_problem):
Three missionaries and three infidels have to cross a river in order to get to a church where the infidels can be baptized.  In order to cross the river, they have to take a small boat that can take at most two passengers.  If at any time the infidels outnumber the missionaries on one shore of the river, the missionaries are in danger.

We will encode this problem as a *constraint satisfaction problem*.  In order to do so, we assume that the
problem can be solved with $n\in\mathbb{N}$ crossings of the river.  We use the following variables:
* $\texttt{M}i$ for $i\in\{0,\cdots,n\}$ is the number of missionaries on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{C}i$ for $i\in\{0,\cdots,n\}$ is the number of infidels on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{MB}i$ for $i\in\{1,\cdots,n\}$ is the number of missionaries that cross the river
  during the $i^{\textrm{th}}$ crossing.
* $\texttt{CB}i$ for $i\in\{1,\cdots,n\}$ is the number of infidels that cross the river
  during the $i^{\textrm{th}}$ crossing.  

We import the backtracking solver.  The notebook `02-Backtracking-Constraint-Solver.ipynb` defines the types `Constraint<'V>` and `CSP<'V>`, the functions `constr` and `hasValue` that create constraints, and the function `solve`.

In [2]:
importNotebook "02-Backtracking-Constraint-Solver.ipynb"

imported 

02-Backtracking-Constraint-Solver.ipynb

## Auxiliary Functions

The function `noProblem` takes one argument:
- `i` is the number of a crossing.

It returns a constraint that is true if there is no problem on either shore of the river after the $i^{\textrm{th}}$ crossing.  There is no problem if any of the following conditions is true:
* There are no missionaries on the western side of the shore, i.e. 
  $\texttt{M}i = 0$.  
  Then all missionaries are on the eastern side of the shore.
* All missionaries are on the western side of the shore, i.e. $\texttt{M}i = 3$.
  Then there are no missionaries on the eastern side of the shore.
* The number of missionaries on the western side is the same as the number of 
  infidels on that side, i.e. $\texttt{M}i = \texttt{C}i$.  Then the numbers of 
  missionaries and infidels have to match on the eastern shore as well.

In [3]:
let noProblem (i: int) : Constraint<int> =
    let M, C = sprintf "M%d" i, sprintf "C%d" i
    constr (sprintf "%s = 0 ∨ %s = 3 ∨ %s = %s" M M M C) [M; C]
           (fun A -> A[M] = 0 || A[M] = 3 || A[M] = A[C])

In [4]:
noProblem 7

M7 = 0 ∨ M7 = 3 ∨ M7 = C7

The function `boatOK` takes one argument:
- `i` is the number of a crossing.

It returns a constraint that is true if the boat is neither empty nor overloaded during the $i^{\textrm{th}}$ crossing, i.e. the number of passengers is either 1 or 2.

In [5]:
let boatOK (i: int) : Constraint<int> =
    let MB, CB = sprintf "MB%d" i, sprintf "CB%d" i
    constr (sprintf "1 ≤ %s + %s ≤ 2" MB CB) [MB; CB]
           (fun A -> 1 <= A[MB] + A[CB] && A[MB] + A[CB] <= 2)

In [6]:
boatOK 3

1 ≤ MB3 + CB3 ≤ 2

The function `changeOK` takes one argument:
- `i` is the number of the state before the crossing.

It returns a list of constraints that are true if the persons on the boat travel from one shore to the other during the $(i+1)^{\textrm{th}}$ crossing.  If $i$ is even, the boat travels from the western shore to the eastern shore and the number of persons on the western shore decreases.  If $i$ is odd, the boat travels back and the number of persons on the western shore increases.

In [7]:
let changeOK (i: int) : Constraint<int> list =
    let M0, M1, MB = sprintf "M%d" i, sprintf "M%d" (i+1), sprintf "MB%d" (i+1)
    let C0, C1, CB = sprintf "C%d" i, sprintf "C%d" (i+1), sprintf "CB%d" (i+1)
    if i % 2 = 1 then
        [ constr (sprintf "%s = %s + %s" M1 M0 MB) [M0; M1; MB] (fun A -> A[M1] = A[M0] + A[MB])
          constr (sprintf "%s = %s + %s" C1 C0 CB) [C0; C1; CB] (fun A -> A[C1] = A[C0] + A[CB]) ]
    else
        [ constr (sprintf "%s = %s - %s" M1 M0 MB) [M0; M1; MB] (fun A -> A[M1] = A[M0] - A[MB])
          constr (sprintf "%s = %s - %s" C1 C0 CB) [C0; C1; CB] (fun A -> A[C1] = A[C0] - A[CB]) ]

In [8]:
changeOK 0

[M1 = M0 - MB1; C1 = C0 - CB1]

In [9]:
changeOK 1

[M2 = M1 + MB2; C2 = C1 + CB2]

The function `missionariesCSP` takes one argument:
- `n` is the number of crossings.

It creates a CSP that tries to solve the problem with `n` crossings.  The variables `MB0` and `CB0` are not needed, since there is no crossing number $0$.

In [10]:
let missionariesCSP (n: int) : CSP<int> =
    let Variables = List.concat [ for i in 0 .. n -> [ sprintf "MB%d" i; sprintf "CB%d" i; sprintf "M%d" i; sprintf "C%d" i ] ]
    { Variables   = List.skip 2 Variables   // remove MB0 and CB0
      Values      = [ 0; 1; 2; 3 ]
      Constraints = [ hasValue "M0" 3; hasValue "C0" 3; hasValue (sprintf "M%d" n) 0; hasValue (sprintf "C%d" n) 0 ]
                    @ [ for i in 0 .. n -> noProblem i ]
                    @ [ for i in 1 .. n -> boatOK i ]
                    @ List.concat [ for i in 0 .. n - 1 -> changeOK i ] }

In [11]:
missionariesCSP 1

{ Variables = ["M0"; "C0"; "MB1"; "CB1"; "M1"; "C1"]; Values = [0; 1; 2; 3]; Constraints = [M0 = 3; C0 = 3; M1 = 0; C1 = 0; M0 = 0 ∨ M0 = 3 ∨ M0 = C0; M1 = 0 ∨ M1 = 3 ∨ M1 = C1; 1 ≤ MB1 + CB1 ≤ 2; M1 = M0 - MB1; C1 = C0 - CB1] }

The function `findSolution` takes no arguments.  It computes a solution to the problem of saving the infidels by trying $n = 1, 3, 5, \cdots$ crossings until a solution is found.  It returns the number of crossings `n` together with the solution.

In [12]:
let findSolution () : int * Map<string, int> =
    let rec search (n: int) =
        printfn "%d" n
        match solve (missionariesCSP n) with
        | Some solution -> (n, solution)
        | None          -> search (n + 2)
    search 1

In [13]:
#!time
let n, Solution = findSolution ()

1

3

5

7

9

11

Wall time: 281.4806ms

In [14]:
n, Solution

(11, map [("C0", 3); ("C1", 1); ("C10", 2); ("C11", 0); ("C2", 2); ("C3", 0); ("C4", 1); ("C5", 1); ("C6", 2); ("C7", 2); ("C8", 3); ("C9", 1); ("CB1", 2); ("CB10", 1); ("CB11", 2); ("CB2", 1); ("CB3", 2); ("CB4", 1); ("CB5", 0); ("CB6", 1); ("CB7", 0); ("CB8", 1); ("CB9", 2); ("M0", 3); ("M1", 3); ("M10", 0); ("M11", 0); ("M2", 3); ("M3", 3); ("M4", 3); ("M5", 1); ("M6", 2); ("M7", 0); ("M8", 0); ("M9", 0); ("MB1", 0); ("MB10", 0); ("MB11", 0); ("MB2", 0); ("MB3", 0); ("MB4", 0); ("MB5", 2); ("MB6", 1); ("MB7", 2); ("MB8", 0); ("MB9", 0)])

The function `showSolution` takes two arguments:
- `Solution` is a solution of the CSP,
- `n` is the number of crossings.

It prints the solution.  For every $i \in \{0, \cdots, n\}$, it prints the situation after the $i^{\textrm{th}}$ crossing.  The western shore is printed on the left, the eastern shore on the right.  Between two situations, the passengers of the boat are shown together with the direction of the crossing.

In [15]:
let showSolution (Solution: Map<string, int>) (n: int) : unit =
    let rep (k: int) (s: string) = String.replicate k s
    for i in 0 .. n do
        let M = Solution[sprintf "M%d" i]
        let C = Solution[sprintf "C%d" i]
        printfn "%s%s%s%s%s" (rep M "😇") (rep C "🥷") (String.replicate 28 " ") (rep (3 - M) "😇") (rep (3 - C) "🥷")
        if i < n then
            let MB = Solution[sprintf "MB%d" (i+1)]
            let CB = Solution[sprintf "CB%d" (i+1)]
            if i % 2 = 0 then
                printfn "%s>>> %s%s >>>" (String.replicate 12 " ") (rep MB "😇") (rep CB "🥷")
            else
                printfn "%s<<< %s%s <<<" (String.replicate 12 " ") (rep MB "😇") (rep CB "🥷")

In [16]:
showSolution Solution n

😇😇😇

🥷🥷🥷

>>> 

🥷🥷

 >>>

...

>>> 

🥷🥷

 >>>

😇😇😇

🥷🥷🥷